# 5-1 · 두 오류의 값이 다르다 — 비용을 세고 판정 점수를 만든다

**⏱ 60분** · 5주차 교재 1/2 — 두 오류와 비용, 자료 나누기, 모델 점수 → (5-2) 검증에서 판정 기준을 고르고 테스트에서 확인하기

## 🎯 이번 주의 질문

스카니아 트럭의 압축 공기 장치(APS)는 제동과 기어 변경 등에 쓰입니다. 정비 기록을 보고 어떤 고장이 APS 고장인지 판정할 때 두 가지로 틀릴 수 있습니다. 다른 부품 고장을 APS 고장으로 잘못 판정하거나(**오경보**), APS 고장을 놓치는 것(**미탐지**)입니다. 이 자료를 공개한 챌린지는 오경보 한 건에 10, 미탐지 한 건에 500의 비용을 매깁니다.

> **오경보 비용 10, 미탐지 비용 500일 때 어느 점수부터 APS 고장으로 판정할까요?**

분류 모델은 기록마다 APS 고장일 가능성을 0~1 사이의 **점수**로 냅니다. 흔히 0.5 이상이면 APS로 판정하지만, 두 오류의 비용이 50배 다르면 그 기준이 맞는지 따져 봐야 합니다.

이번 시간에는 두 오류를 세어 비용으로 바꾸는 법을 익히고, 자료를 역할별로 나눈 뒤 모델로 점수를 만듭니다. 어느 점수부터 APS로 판정할지는 5-2에서 정합니다.

**데이터:** [UCI APS Failure at Scania Trucks](https://archive.ics.uci.edu/dataset/421/aps+failure+at+scania+trucks) — 실제 트럭 운행 기록 중 전문가가 고른 고장 기록입니다. 학습 파일 60,000건, 테스트 파일 16,000건입니다.

**이번 시간에 반복할 패턴:** 두 오류 세기 `((판정 == 1) & (정답 == 0)).sum()` → 비용 `10 × 오경보 + 500 × 미탐지`

### 셀 표시 읽는 법

| 표시 | 뜻 | 할 일 |
|---|---|---|
| ▶ | 제공 코드 | 그대로 실행하고 **결과만** 읽습니다. 코드를 외울 필요는 없습니다 |
| ✍️ | 직접 입력 | 보여 준 코드를 **손으로 타이핑**해서 실행합니다. 이번 주에 반복하는 핵심 패턴입니다 |
| 📝 | 과제 | 앞에서 입력한 코드를 **조금 바꿔서** 스스로 풉니다 |
| 💬 · ✅ | 결과 해설 · 체크포인트 | 숫자를 읽는 법, 내 결과가 맞는지 확인할 값 |

> **실습본입니다.** 먼저 메뉴 **파일 → Drive에 사본 저장**을 누르세요. 사본을 저장하지 않으면 입력한 코드가 사라집니다.
> 맨 위 준비 셀부터 순서대로 `Shift + Enter`로 실행합니다. ▶ 셀은 그대로 실행하고, ✍️ 셀은 위에 보여 준 코드를
> 빈 셀에 손으로 타이핑하고, 빈칸(`___`)은 채워서 실행합니다. 📝 과제는 힌트를 보고 스스로 풉니다.

In [ ]:
# ▶ 준비 셀 — 그대로 실행하세요. APS 고장 기록의 학습 파일(tr)과 테스트 파일(te)을 읽습니다.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from lightgbm import LGBMClassifier
import os
import sys

for folder in (".", "..", "../..", "../../.."):
    if os.path.exists(os.path.join(folder, "balab.py")):
        sys.path.insert(0, folder)
        break
else:
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/BALAB-PKNU/bizanalytics/main/balab.py", "balab.py")

from balab import load
aps = load("aps")
tr = aps["train"]
te = aps["test"]
print("준비 완료!", tr.shape, te.shape)

✅ `준비 완료! (60000, 171) (16000, 171)`이 나오면 됩니다. 테스트 파일 `te`는 5-2의 마지막 평가까지 쓰지 않습니다.

---
## 1. 한 행은 무엇이고, APS 고장은 몇 건인가

학습 파일의 앞부분부터 봅니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
tr.head()
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 **결과 해설** — **한 행 = 트럭 고장 기록 하나**입니다.

| 열 | 뜻 |
|---|---|
| `class` | `pos`는 특정 APS 부품 고장, `neg`는 APS와 관련 없는 부품 고장 |
| `aa_000` 등 170개 수치 열 | 익명화된 누적값과 구간별 집계. 실제 이름은 비공개 |

- 모든 행이 고장 기록이므로, 이 자료의 비교는 **APS 고장과 다른 고장 사이의 분류**입니다. 정상 운행 트럭의 고장률이나 앞으로 며칠 안에 고장 날 확률을 재는 자료가 아닙니다.
- 변수 이름이 가려져 있어 어떤 물리적 현상이 고장을 일으켰는지는 이 자료만으로 설명하지 않습니다.
- 빈 값(`NaN`)은 원본에서 `na`로 적힌 값입니다.

두 종류가 각각 몇 건인지 셉니다. `.value_counts()`는 값마다 개수를 셉니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
tr["class"].value_counts()
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


✅ **체크포인트** — `neg 59000`, `pos 1000`이면 맞습니다.

💬 APS 고장은 60건 중 1건꼴입니다. 모든 기록을 "APS 아님"으로 판정해도 대부분 맞힌다는 뜻이라, 몇 건을 맞혔는지(정확도)만으로는 판정을 평가하기 어렵습니다. 2절에서 두 오류를 따로 셉니다.

모델에 넣을 입력 표 `X`와 정답 `y`를 만듭니다. 입력은 `class`를 뺀 170개 열이고(`.drop(columns=[...])`), 정답은 APS 고장이면 1, 아니면 0입니다(4주차에서 `rain`을 만든 비교 → `astype(int)`). 입력 열마다 빈 값의 비율도 봅니다. `.isna()`는 빈 값이면 참, 참·거짓의 평균은 참의 비율입니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
X = tr.drop(columns=["class"])
y = (tr["class"] == "pos").astype(int)
X.isna().mean().sort_values(ascending=False).head()
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 빈 값이 가장 많은 열 `br_000`은 82%가 비어 있습니다. 빈 값은 4절에서 **모델을 학습하는 자료의 중앙값**으로 채웁니다.

---
## 2. 두 오류와 비용

판정이 틀리는 경우는 두 가지입니다.

| | 정답 APS(1) | 정답 다른 고장(0) |
|---|---|---|
| **APS로 판정(1)** | 맞힘 | **오경보** — 비용 10 |
| **다른 고장으로 판정(0)** | **미탐지** — 비용 500 | 맞힘 |

맞힌 판정의 비용은 0으로 둡니다. 10과 500은 실제 수리비 전체가 아니라 두 오류에 매긴 상대적 벌점입니다. 총비용은 `10 × 오경보 수 + 500 × 미탐지 수`입니다.

기록 네 개짜리 연습용 표시로 손 계산부터 해 봅니다. 정답이 0인데 1로 판정한 것이 오경보, 정답이 1인데 0으로 판정한 것이 미탐지입니다. 두 조건을 모두 만족하는 행은 `&`로 묶고, 참의 개수는 `.sum()`으로 셉니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
answer = pd.Series([0, 1, 1, 0])
decision = pd.Series([1, 0, 1, 0])
fp_small = ((decision == 1) & (answer == 0)).sum()
fn_small = ((decision == 0) & (answer == 1)).sum()
fp_small, fn_small, 10 * fp_small + 500 * fn_small
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


✅ **체크포인트** — `(1, 1, 510)`이면 맞습니다.

💬 첫 행(정답 0, 판정 1)이 오경보, 둘째 행(정답 1, 판정 0)이 미탐지입니다. 틀린 개수는 한 건씩 같아도 비용은 10과 500으로 다릅니다.

이 계산을 이번 주 내내 여러 판정에 되풀이하므로 **함수**로 묶어 둡니다. `def 이름(입력):`으로 함수를 만들고, `return`으로 결과를 돌려줍니다. 두 값을 돌려주면 `fp, fn = errors(...)`처럼 두 변수로 나눠 받을 수 있습니다. `int(...)`는 개수를 보통 정수로 바꿉니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
def errors(actual, predicted):
    fp = int(((predicted == 1) & (actual == 0)).sum())
    fn = int(((predicted == 0) & (actual == 1)).sum())
    return fp, fn

errors(answer, decision)
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 `(1, 1)` — 손으로 센 오경보·미탐지와 같습니다. 이제 어떤 판정이든 `errors(정답, 판정)` 한 줄로 두 오류를 셉니다.

---
## 3. 자료를 세 역할로 나눈다

판정 기준을 고른 자료로 그 기준의 성적을 다시 내면, 그 자료에 맞춰 고른 만큼 성적이 좋게 나옵니다. 그래서 자료를 역할별로 나눕니다.

| 자료 | 크기 | 하는 일 |
|---|---|---|
| 학습용 | 48,000건 | 빈 값을 채울 중앙값과 모델을 만든다 |
| 검증용 | 12,000건 | 판정 기준을 고른다 (5-2) |
| 테스트 | 16,000건 (별도 파일) | 고정한 기준을 마지막에 평가한다 (5-2) |

학습 파일 60,000건을 48,000건과 12,000건으로 나눕니다. `train_test_split`은 자료를 무작위로 둘로 나누는 함수입니다. `test_size`에 떼어 낼 건수, `stratify`에 정답을 주면 두 쪽의 APS 비율이 같게 나뉘고(**층화 분할**), `random_state`는 매번 같은 결과가 나오게 하는 번호입니다. 결과는 학습 입력, 검증 입력, 학습 정답, 검증 정답 순서로 받습니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
X_fit, X_val, y_fit, y_val = train_test_split(
    X, y, test_size=12000, stratify=y, random_state=0)
len(X_fit), len(X_val), int(y_fit.sum()), int(y_val.sum())
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


✅ **체크포인트** — `(48000, 12000, 800, 200)`이면 맞습니다.

💬 학습용 48,000건에 APS 800건, 검증용 12,000건에 APS 200건입니다. 두 쪽 모두 60건 중 1건꼴입니다.

모델 없이 낼 수 있는 가장 단순한 두 판정의 비용부터 봅니다. 모두 "APS 아님"(0)으로 판정하면 APS 200건을 모두 놓치고, 모두 "APS"(1)로 판정하면 나머지 11,800건이 모두 오경보입니다. 정답과 길이가 같은 0 판정은 `y_val * 0`, 1 판정은 `y_val * 0 + 1`로 만듭니다. **두 오류 세기 → 비용** 패턴을 두 번 씁니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
fp_none, fn_none = errors(y_val, y_val * 0)
fp_all, fn_all = errors(y_val, y_val * 0 + 1)
10 * fp_none + 500 * fn_none, 10 * fp_all + 500 * fn_all
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


✅ **체크포인트** — `(100000, 118000)`이면 맞습니다.

💬 **결과 해설**
- 전원 "APS 아님": 미탐지 200건 × 500 = **100,000**
- 전원 "APS": 오경보 11,800건 × 10 = **118,000**
- 모델로 만든 판정은 이 두 값보다 훨씬 낮아야 쓸모가 있습니다. 5-2에서 이 두 판정도 후보에 넣어 함께 비교합니다.

---
## 4. 모델로 판정 점수를 만든다

빈 값을 채울 중앙값도 모델의 일부처럼 **학습용에서만** 구합니다. 검증용에서 중앙값을 다시 구하면 검증 자료의 정보가 학습에 섞입니다. `.median()`은 열마다 중앙값을, `.fillna(값)`은 빈 칸을 그 값으로 채웁니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
med = X_fit.median()
X_fit = X_fit.fillna(med)
X_val = X_val.fillna(med)
X_fit.isna().sum().sum(), X_val.isna().sum().sum()
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 `(0, 0)` — 두 입력 모두 빈 값이 남지 않았습니다.

모델은 **LightGBM**을 씁니다. 여러 결정 트리를 순서대로 더해 가며 앞의 트리가 틀린 부분을 고치는 모델입니다. 트리 수·잎 수·학습률 같은 설정은 아래 값으로 고정해 두었으므로 실행만 합니다. 이번 주에 고를 것은 모델 설정이 아니라 **점수를 판정으로 바꾸는 기준**입니다.

In [ ]:
# ▶ 그대로 실행하세요 — 고정 설정의 LightGBM 모델 (설정 뜻은 배경지식 6절)
model = LGBMClassifier(
    num_leaves=15, learning_rate=0.03, n_estimators=800,
    min_child_samples=20, subsample=0.8, subsample_freq=1,
    colsample_bytree=0.8, verbose=-1, random_state=0,
    n_jobs=1, deterministic=True, force_col_wise=True)

**모델 패턴** — 머신러닝 모델은 `model.fit(입력, 정답)`으로 학습합니다. 학습용 입력과 정답을 줍니다. 30초~1분쯤 걸립니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
model.fit(X_fit, y_fit)
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


학습한 모델은 `.predict_proba(입력)`으로 기록마다 두 열의 점수를 냅니다. 어느 열이 APS인지는 `.classes_`의 순서로 확인합니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
model.classes_
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 순서가 `[0, 1]`이므로 둘째 열이 APS(1)의 점수입니다. 0부터 세므로 열 번호는 1입니다.

검증 기록에 점수를 매깁니다. 배열에서 `[:, 1]`은 "모든 행의 1번 열"입니다. 점수를 검증 입력과 같은 행 번호(`index`)의 Series로 만들어 두면 정답 `y_val`과 행이 맞습니다. `.describe()`로 분포를 요약합니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
prob_val = model.predict_proba(X_val)
p_val = pd.Series(prob_val[:, 1], index=X_val.index)
p_val.describe()
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


✅ **체크포인트** — `count 12000`, `50%`가 `1.49e-05` 근처면 맞습니다.

💬 **결과 해설**
- 점수는 0~1 사이이고, 중앙값은 **0.0000149**, 75% 지점도 0.0000463입니다. 대부분의 기록이 0에 아주 가깝습니다.
- 가장 높은 점수는 0.99994입니다. APS가 60건 중 1건이니, 점수가 높은 기록은 소수입니다.
- 이 점수는 모델이 추정한 확률이지만, 실제 APS 비율과 딱 맞는다는 보장은 없습니다. 그래서 5-2에서는 확률값을 그대로 믿기보다 **검증 자료의 비용**으로 판정 기준을 고릅니다.

---
## 5. 실습 과제 (15분)

### 📝 과제 1 · 정확도는 높은데 비용은 큰 판정

3절의 두 극단 판정의 검증 **정확도**를 구하세요. 정확도는 판정과 정답이 같은 기록의 비율입니다.

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: 판정과 y_val이 같은지 비교하면 기록마다 참·거짓이 나오고, 그 평균이 비율입니다(4주차에서 rain의 평균이 비율이었던 것과 같습니다). 판정은 3절처럼 y_val * 0과 y_val * 0 + 1을 씁니다.


✏️ **나의 답:** 전원 "APS 아님" 판정의 정확도와 비용을 함께 보고, 이번 주에 정확도로 판정 기준을 고르면 안 되는 이유를 한두 문장으로 적습니다. (여기를 두 번 눌러 적습니다)

### 📝 과제 2 · APS 기록과 다른 고장 기록의 점수는 얼마나 다른가

검증 기록을 정답에 따라 둘로 나눠 점수 `p_val`의 중앙값을 각각 구하세요.

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: 조건 필터로 y_val이 1인 기록의 점수만 고르고 median을 구합니다. y_val이 0인 기록도 같은 방법입니다.


✏️ **나의 답:** 두 집단의 점수는 얼마나 다른가요? 모든 APS 기록이 높은 점수를 받았다고 말할 수 있을까요?

### 📝 과제 3 · 생각해 보기

판정 기준을 검증 자료로 고른 뒤 같은 검증 자료로 그 기준의 비용을 보고하면 무엇이 문제일까요? 테스트 파일을 마지막까지 쓰지 않는 이유와 함께 적으세요.

✏️ **나의 답:**

---
## ✅ 정리

| 배운 것 | 핵심 |
|---|---|
| 한 행의 의미 | 트럭 고장 기록 하나. APS 고장 1,000건, 다른 고장 59,000건(60건 중 1건) |
| 두 오류 | 오경보(다른 고장 → APS, 비용 10), 미탐지(APS → 놓침, 비용 500). 틀린 개수가 같아도 비용은 다르다 |
| 두 오류 세기 | `errors(정답, 판정)` → `10 * fp + 500 * fn` |
| 자료의 역할 | 학습용 48,000건(중앙값·모델) · 검증용 12,000건(기준 고르기) · 테스트 16,000건(마지막 평가) |
| 두 극단 | 전원 "APS 아님" 100,000 · 전원 "APS" 118,000. 모델 판정은 이보다 낮아야 한다 |
| 판정 점수 | `model.fit` → `predict_proba(...)[:, 1]`. 대부분 0에 가깝고 소수만 높다 |

다음: **5-2 · 어느 점수부터 APS로 판정할까 — 검증에서 고르고 테스트에서 확인**